# Lesson 05 — Vector Search

We have frame embeddings in S3 Vectors from Lesson 04. Now we search them **by text**.

The idea:
1. Encode a text query with CLIP → get a 512-d vector
2. Ask S3 Vectors for the nearest frame vectors
3. Return the top-k matching frames and timestamps

No Batch job this lesson — CLIP's text encoder runs locally on CPU and S3
Vectors performs the nearest-neighbour search.

> **No local GPU?** That's fine. CLIP's text encoder is small and runs on CPU in <1 second.

## Step 1 — Load environment and connect to S3 Vectors

In [ ]:
import io, os
import boto3
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]
S3_VECTOR_BUCKET = os.environ["S3_VECTOR_BUCKET"]
S3_VECTOR_INDEX = os.environ["S3_VECTOR_INDEX"]

s3 = boto3.client("s3")
s3vectors = boto3.client("s3vectors")

print(f"Frame bucket : {S3_BUCKET}")
print(f"Vector index : {S3_VECTOR_BUCKET}/{S3_VECTOR_INDEX}")

## Step 2 — Load the CLIP text encoder

In [ ]:
import clip, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model, _ = clip.load("ViT-B/32", device=device)
model.eval()
print("CLIP loaded.")

## Step 3 — Search helper

This function encodes a text query, then asks S3 Vectors for the top matching frames.

In [ ]:
def search(query: str, top_k: int = 3) -> list[dict]:
    """Return the top_k S3 Vectors matches for a plain-English query."""
    # Encode the query text
    tokens       = clip.tokenize([query]).to(device)
    with torch.no_grad():
        text_vec = model.encode_text(tokens)           # shape: (1, 512)
        text_vec = text_vec / text_vec.norm(dim=-1, keepdim=True)   # normalise

    response = s3vectors.query_vectors(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=S3_VECTOR_INDEX,
        topK=top_k,
        queryVector={"float32": text_vec.cpu().numpy().flatten().astype("float32").tolist()},
        returnMetadata=True,
        returnDistance=True,
    )
    return response["vectors"]

print("Search function ready.")

## Step 4 — Search! (edit the query and re-run)

Try: `"outdoor scene"`, `"close-up face"`, `"text on screen"`, `"dark scene"`, `"a busy street"`

In [ ]:
QUERY = "outdoor scene with trees"   # ← change me!

matches = search(QUERY, top_k=3)

print(f"Query: '{QUERY}'")
for rank, match in enumerate(matches, start=1):
    metadata = match["metadata"]
    print(f"  #{rank}  {metadata['frame_key']}  time={metadata['timestamp_ms'] / 1000:.1f}s  distance={match['distance']:.3f}")

## Step 5 — Show the top-3 matching frames

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, match in zip(axes, matches):
    metadata = match["metadata"]
    key = metadata["frame_key"]
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    img = Image.open(io.BytesIO(obj["Body"].read()))
    ax.imshow(img)
    ax.set_title(f"{metadata['timestamp_ms'] / 1000:.1f}s\nDistance: {match['distance']:.3f}", fontsize=9)
    ax.axis("off")

plt.suptitle(f"Top-3 results for: '{QUERY}'", fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaway

> We searched through all frames using **plain English**, with no labels or
> keyword index. The GPU created the CLIP embeddings and S3 Vectors retrieved
> the nearest matching frames.

This pattern — embed → store → search by cosine similarity — powers semantic search, RAG, and image search at every major tech company.

---

## Next lesson → [06 — Full Pipeline](../06-full-pipeline/notebook.ipynb)

We'll string lessons 03 and 04 together with Batch job dependencies — one command runs the full pipeline.